# Final report on data challenge "Maintenance and Industry 4.0 2026"

## Group X - Members

- Óscar Martínez Zamora
- Yiling ZHUANG
- Ahmed-Wassim BENZERGA

## Submission
- Please name your notebook as `final_submission_group_X.ipynb` and submit it via [here](https://nextcloud.centralesupelec.fr/s/bcGcQzimE4ZjetS)
- Deadline: 07/06/2026 23:59

## Methods

### Overview:
Instead of building a one-size-fits-all model, we treated this as **six separate binary classification tasks** (one for each motor). We did this because the motors behave entirely differently. For instance, Motors 2 and 4 fail about 17% of the time, while Motors 3 and 5 barely fail at all (under 0.5%). All six motors run through the exact same pipeline, but we fine-tuned the settings for each one individually.

### How We Trained It
We used the main labeled training data, the supplementary Kaggle datasets, and some synthetic data we generated ourselves (more on that below). 

What needs to be taken care of here was data leakage. Because data points close to each other in time are practically identical, doing a standard random row-wise split would give us a falsely high F1 score that would crash and burn on the actual Kaggle leaderboard. To fix this, we kept our validation strictly **sequence-by-sequence**. 

### Creating Synthetic Faults
Motors 1, 3, and 5 just didn't have enough real failures for the model to learn what a fault looks like. So, we injected some failures using the code from Kaggle. We took normal, healthy sequences and spiked the temperature data with a simulated "rise-and-fall" pulse, labeling that window as a fault. We tweaked how aggressive and frequent these fake pulses were depending on the motor—the motors that rarely fail got more frequent, subtler spikes to learn from.

### Feature Engineering 
We noticed a weird thing in the data: the relationship between temperature and faults completely flipped between two of the real sequences. Because of this, just looking at absolute temperature stopped being useful. Instead, we built out features in three stages:

1. **Relative Basics:** We measured everything relative to the start of the sequence to wipe out any baseline temperature differences.
2. **Capturing Change:** We looked at rolling averages, standard deviations, and differences over time (lags). This helped the model focus on *how* the temperature was shifting rather than the raw number, which made it robust against that weird data flip. This was the biggest performance jump for Motors 1 and 4.
3. **Looking at the "Shape" (Motor 1 only):** For Motor 1, we looked at the statistical shape of the temperature spikes—things like skew, kurtosis, and rolling deviations. Since these focus on the *pattern* of a spike rather than its height, Motor 1's performance shot up from a 0.60 to a **0.72**.

### Picking the Model
We tested the usual suspects (Random Forest, Logistic Regression, Isolation Forest, etc.), but **HistGradientBoosting swept the board for all six motors**. It easily handled our mixed feature types, dealt with the imbalanced data perfectly using sample weights, and was fast enough that we could run heavy grid searches for each motor without waiting forever.

### Smoothing Out the Predictions
We customized the decision thresholds for each motor since a default 0.5 cutoff doesn't work well when faults are rare. 

Also, since real machine faults happen over a continuous block of time, we cleaned up the model's raw predictions using morphological post-processing. We patched up tiny gaps to merge fragmented runs, and we deleted random, isolated "blips" that were too short to be real. This cleanup step gave us massive boosts—for example, Motor 4 jumped from 0.66 to **0.90**, and Motor 6 went from 0.89 to **0.93**.
**Final per-motor models (best configuration):**

The *held-out F1* column is the fault-class F1 on a per-sequence validation split (computed below, a
**conservative** internal estimate). The *Kaggle* column is the public leaderboard F1 of that motor in our
best submission. The two differ —especially for the rare-fault motors— because the local split is tiny and
the test thresholds were refined per motor; this gap is discussed at the end.

| Motors | Final model | Features | Dataset used for training | Held-out F1 / **Kaggle (public)** | Additional notes |
| --- | --- | --- | --- | --- | --- |
| Motor 1 | HistGradientBoosting | dynamic + **window-shape** (skew/kurt/slope/range/EWMA) | training + supplementary + synthetic (peak 8–16°C ×4) | 0.47 / **0.72** | hardest: temp↔fault inversion; window-shape moments help |
| Motor 2 | HistGradientBoosting | physical + dynamic + rolling | training + supplementary + synthetic (peak 2–10°C ×4) | 0.43 / **0.90** | frequent fault, well separable |
| Motor 3 | HistGradientBoosting | physical + dynamic + rolling | training + supplementary + synthetic (peak 3–8°C ×16) | 0.95 / **0.77** | <0.5% faults; very fragile (see discussion) |
| Motor 4 | HistGradientBoosting | dynamic | training + supplementary + synthetic (peak 6–15°C ×4) | 0.72 / **0.90** | inversion handled by dynamics + post-proc (close=5, min_run=40) |
| Motor 5 | HistGradientBoosting | physical + dynamic | training + supplementary + synthetic (peak 1–4°C ×16) | 0.76 / **0.96** | <0.5% faults; subtle pulse |
| Motor 6 | HistGradientBoosting | physical + dynamic | training + supplementary + synthetic (peak 6–15°C ×4) | 0.89 / **0.93** | post-proc min_run=20 |

**Best Kaggle macro-F1 (public): 0.866** — per motor: M1 0.722 · M2 0.905 · M3 0.769 · M4 0.901 · M5 0.963 · M6 0.934.

## Results

We first build the **shared pipeline** (loading, preprocessing, feature engineering, synthetic injection,
model factory, post-processing). Each motor's subsection then states its tried models and the chosen one;
a single training cell fits all six with their per-motor settings and reports the held-out F1. The last
subsection uses those models to produce the Kaggle submission file.

### Shared pipeline (used by all motors)

The code below is self-contained and uses the standard labelled training data so the notebook is fully
reproducible end-to-end.

In [1]:
import os, sys, itertools, warnings
import numpy as np
import pandas as pd
from scipy.ndimage import binary_closing
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

ROOT = os.getcwd()
sys.path.insert(1, os.path.join(ROOT, 'kaggle_data_challenge'))
from utility import read_all_test_data_from_path   # course-provided loader

ROLL_WINDOWS, WIN_SHAPE_WINDOWS, EWMA_SPANS = [5, 20, 50], [20, 50], [10, 30]

def _rolling_slope(temp, w):
    x = np.asarray(temp, dtype=float); n = len(x)
    if n < 2 or w < 2: return pd.Series(np.zeros(n), index=temp.index)
    t = np.arange(w, dtype=float); t -= t.mean(); denom = (t*t).sum()
    conv = np.convolve(x, t[::-1]/denom, mode='full')[:n]; conv[:w-1] = 0.0
    return pd.Series(conv, index=temp.index)

def temp_derived_features(temp):
    out = {}
    out['temperature_diff']    = temp.diff(20)
    out['temperature_diff_5']  = temp.diff(5)
    out['temperature_diff_50'] = temp.diff(50)
    for w in ROLL_WINDOWS:
        out[f'temperature_roll_mean_{w}'] = temp.rolling(w, min_periods=1).mean()
        out[f'temperature_roll_max_{w}']  = temp.rolling(w, min_periods=1).max()
        out[f'temperature_roll_std_{w}']  = temp.rolling(w, min_periods=1).std().fillna(0)
    out['temperature_dev_20'] = temp - temp.rolling(20, min_periods=1).mean()
    for w in WIN_SHAPE_WINDOWS:                     # window-shape (M1)
        r = temp.rolling(w, min_periods=2); slope = _rolling_slope(temp, w)
        out[f'temperature_roll_skew_{w}']  = r.skew().fillna(0)
        out[f'temperature_roll_kurt_{w}']  = r.kurt().fillna(0)
        out[f'temperature_roll_range_{w}'] = (temp.rolling(w, min_periods=1).max()
                                              - temp.rolling(w, min_periods=1).min())
        out[f'temperature_slope_{w}']    = slope
        out[f'temperature_absslope_{w}'] = slope.abs()        # direction-invariant
    for span in EWMA_SPANS:
        e = temp.ewm(span=span, adjust=False).mean()
        out[f'temperature_ewma_{span}']     = e
        out[f'temperature_ewma_dev_{span}'] = temp - e
    return out

def pre_processing(df):
    if len(df) == 0: return
    df['temperature'] = df['temperature'].where(df['temperature'].between(0, 100)).ffill()
    df['voltage']     = df['voltage'].where(df['voltage'].between(6000, 9000)).ffill()
    df['position']    = df['position'].where(df['position'].between(0, 1000)).ffill()
    for col in ['temperature', 'voltage', 'position']:
        df[col] -= df[col].iloc[0]                  # relative to first sample
    df['voltage_diff']  = df['voltage'].diff(20)
    df['position_diff'] = df['position'].diff(20)
    for name, s in temp_derived_features(df['temperature']).items():
        df[name] = s
    df.fillna(0, inplace=True)

print('Pipeline helpers defined.')

Pipeline helpers defined.


In [2]:
# ---- Load training & test data (per-sequence), then add movement-description features ----
import shutil
train_df = read_all_test_data_from_path(os.path.join(ROOT, 'data', 'training_data') + os.sep,
                                        pre_processing, is_plot=False)
test_df  = read_all_test_data_from_path(os.path.join(ROOT, 'data', 'testing_data') + os.sep,
                                        pre_processing, is_plot=False)

# Supplementary training datasets shared during the challenge (filter out empty subdirs).
def load_additional(base, groups):
    dfs = []
    for g in groups:
        gp = os.path.join(base, g)
        xlsx = os.path.join(gp, 'Test conditions.xlsx')
        if not os.path.exists(xlsx) and os.path.exists(os.path.join(gp, 'Test conditions copy.xlsx')):
            shutil.copy(os.path.join(gp, 'Test conditions copy.xlsx'), xlsx)
        if not os.path.isdir(gp): continue
        tmp = os.path.join(base, f'_nb_tmp_{g}'); shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp, exist_ok=True)
        if os.path.exists(xlsx): shutil.copy(xlsx, os.path.join(tmp, 'Test conditions.xlsx'))
        for sub in os.listdir(gp):
            sp = os.path.join(gp, sub)
            if not os.path.isdir(sp): continue
            csvs = [f for f in os.listdir(sp) if f.endswith('.csv')]; ok = bool(csvs)
            for cf in csvs:
                try:
                    with open(os.path.join(sp, cf)) as fh: n = sum(1 for l in fh if l.strip())
                    if n <= 1: ok = False; break
                except Exception: ok = False; break
            if ok: shutil.copytree(sp, os.path.join(tmp, sub), dirs_exist_ok=True)
        try:
            dfs.append(read_all_test_data_from_path(tmp + os.sep, pre_processing, is_plot=False))
        except Exception as e:
            print('  (skip', g, '->', e, ')')
        shutil.rmtree(tmp, ignore_errors=True)
    return dfs

extra = load_additional(os.path.join(ROOT, 'data', 'additional_data'),
                        ['additional_data_20240524_group_6', 'additional_training_data_group_1',
                         'additional_training_data_group_7'])
if extra:
    train_df = pd.concat([train_df] + extra, ignore_index=True)
print('train:', train_df.shape, '| test:', test_df.shape)

import glob
DESC = ['desc_transfer', 'desc_not_moving', 'desc_turn_motor', 'desc_chute_cube']
tc = pd.concat([pd.read_excel(f) for f in glob.glob(os.path.join(ROOT, 'data') + '/**/Test conditions*.xlsx',
               recursive=True)], ignore_index=True)
tc['Description'] = tc['Description'].fillna('').astype(str).str.lower()
desc_map = tc.set_index('Test id')['Description'].to_dict()
for df in (train_df, test_df):
    raw = df['test_condition'].map(desc_map).fillna('')
    flags = {'desc_transfer': 'transfer', 'desc_not_moving': 'not moving',
             'desc_turn_motor': 'turn motor', 'desc_chute_cube': 'chute cube'}
    new = {f'data_motor_{m}_{f}': raw.str.contains(k).astype(int).values
           for m in range(1, 7) for f, k in flags.items()}
    for c, v in new.items(): df[c] = v
print('Description features added.')

train: (99407, 200) | test: (14157, 200)


Description features added.


In [3]:
# ---- Feature sets, injection and post-processing settings (per motor) ----
DESC_FEATS = [f for f in DESC]
ROLL_V2 = ['temperature_roll_mean_5','temperature_roll_max_5','temperature_roll_std_5',
           'temperature_roll_mean_20','temperature_roll_max_20','temperature_roll_std_20']
DYN = ['temperature_diff','temperature_diff_5','temperature_diff_50','temperature_roll_std_5',
       'temperature_roll_std_20','temperature_roll_std_50','temperature_dev_20',
       'temperature_roll_max_20','position','voltage'] + DESC_FEATS
WIN = ['temperature_roll_skew_20','temperature_roll_kurt_20','temperature_roll_range_20',
       'temperature_slope_20','temperature_roll_skew_50','temperature_roll_kurt_50',
       'temperature_roll_range_50','temperature_slope_50','temperature_ewma_dev_10',
       'temperature_ewma_dev_30']

MOTOR_FEATURES = {
    1: DYN + WIN,                                                         # dynamic + window-shape
    2: ['temperature','position','voltage'] + ROLL_V2 + DESC_FEATS,
    3: ['position','temperature_diff','voltage','temperature'] + ROLL_V2 + DESC_FEATS,
    4: DYN,                                                               # dynamic
    5: ['position','voltage','temperature','temperature_diff'] + ROLL_V2 + DESC_FEATS,
    6: ['position','temperature','temperature_diff','position_diff'] + ROLL_V2 + DESC_FEATS,
}
N_INJECT = {1: 4, 2: 4, 3: 16, 4: 4, 5: 16, 6: 4}
PEAK     = {1: (8,16), 2: (2,10), 3: (3,8), 4: (6,15), 5: (1,4), 6: (6,15)}
POST     = {4: dict(close=5, min_run=40), 6: dict(close=None, min_run=20)}   # else default closing
HGB_PARAMS = dict(learning_rate=0.05, max_iter=200, max_depth=4, min_samples_leaf=200, random_state=42)

def inject_failure(temp, label, peak_low, peak_high):
    temp = np.asarray(temp, float).copy(); label = np.asarray(label); mask = label == 1
    tmp = temp[mask].copy(); n = len(tmp)
    if n == 0: return temp
    nr = max(1, n // 3); ts, te = tmp[0], tmp[-1]
    th = max(ts, te) + np.random.randint(peak_low, peak_high + 1)
    rs = int(round(th - ts))
    if rs >= 1:
        st = (nr // (rs + 1)) or 1; i = 0
        for i in range(1, rs + 1):
            lo, hi = (i-1)*st, min(i*st, nr)
            if lo >= nr: break
            tmp[lo:hi] = ts + (i-1)
        if i*st < nr: tmp[i*st:nr] = th
    ds = int(round(th - te))
    if ds >= 1:
        st = (2*nr // ds) or 1; i = 0
        for i in range(1, ds):
            lo, hi = nr+(i-1)*st, min(nr+i*st, n)
            if lo >= n: break
            tmp[lo:hi] = th - i
        if nr+i*st < n: tmp[nr+i*st:] = te
    temp[mask] = tmp; return temp

def synthesize(raw, mid, n_seq, peak):
    lab = f'data_motor_{mid}_label'; tcol = f'data_motor_{mid}_temperature'; pcol = f'data_motor_{mid}_position'
    normals = [s for s, g in raw.groupby('test_condition', sort=False) if (g[lab] == 1).sum() == 0]
    if not normals: return pd.DataFrame()
    frames, made, att = [], 0, 0
    while made < n_seq and att < n_seq*20:
        att += 1; src = np.random.choice(normals); seq = raw[raw['test_condition'] == src]; n = len(seq)
        emax = min(400, n-10)
        if emax < 120: continue
        flen = np.random.randint(120, emax+1); start = np.random.randint(0, max(1, n-flen))
        out = seq.copy().reset_index(drop=True); lbl = np.zeros(len(out), int); lbl[start:start+flen] = 1
        out[tcol] = inject_failure(out[tcol].to_numpy(), lbl, *peak); out[lab] = lbl
        for j in range(1, 7):
            if j != mid: out[f'data_motor_{j}_label'] = np.nan
        t = out[tcol]
        if pcol in out.columns: out[f'data_motor_{mid}_position_diff'] = out[pcol].diff(20).fillna(0)
        for name, s in temp_derived_features(t).items(): out[f'data_motor_{mid}_{name}'] = s.fillna(0).values
        out['test_condition'] = f'{src}_synth_{mid}_{made}'; frames.append(out); made += 1
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def remove_short_runs(y, k):
    y = np.asarray(y, int).copy()
    if k <= 1: return y
    i, n = 0, len(y)
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1: j += 1
            if j - i < k: y[i:j] = 0
            i = j
        else: i += 1
    return y
print('Per-motor settings and injection/post-processing defined.')

Per-motor settings and injection/post-processing defined.


In [4]:
# ---- Train one motor: per-sequence split, inject, GRID-SEARCH HistGB, pick threshold on val ----
PARAM_GRID = [dict(zip(['learning_rate','max_iter','max_depth','min_samples_leaf'], v))
              for v in itertools.product([0.02,0.05],[100,150],[3,4,5],[200,600])]

def _split(lst):
    if len(lst) > 1:
        return train_test_split(lst, test_size=max(1,int(0.2*len(lst))), random_state=42)
    return lst, []

THRESHOLDS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

def train_motor(mid, seed=42):
    np.random.seed(seed*1000 + mid)
    feats = [f'data_motor_{mid}_{f}' for f in MOTOR_FEATURES[mid]]
    lab = f'data_motor_{mid}_label'
    mdf = train_df.dropna(subset=[lab]).copy()
    seqs = mdf['test_condition'].unique()
    fail = [s for s in seqs if (mdf[mdf.test_condition==s][lab]==1).any()]
    norm = [s for s in seqs if s not in fail]
    trf, vaf = _split(fail); trn, van = _split(norm)
    tr_seqs, va_seqs = trf+trn, vaf+van
    raw_tr = mdf[mdf.test_condition.isin(tr_seqs)].copy()
    synth = synthesize(raw_tr, mid, N_INJECT[mid], PEAK[mid])
    aug = pd.concat([raw_tr, synth], ignore_index=True) if not synth.empty else raw_tr
    Xtr, ytr = aug[feats], aug[lab]; sw = np.where(ytr==1, 2.0, 1.0)
    pp = POST.get(mid, {}); close_size = pp.get('close', None); mr = pp.get('min_run', 0)
    va = mdf[mdf.test_condition.isin(va_seqs)] if va_seqs else None

    # Stage 1: choose hyperparameters by best THRESHOLD-ONLY F1 (no closing).
    best_params, best_score, model0 = PARAM_GRID[0], -1.0, None
    for params in PARAM_GRID:
        m = HistGradientBoostingClassifier(random_state=42, **params).fit(Xtr, ytr, sample_weight=sw)
        if va is not None and va[lab].nunique() > 1:
            p = m.predict_proba(va[feats])[:, 1]
            sc = max(f1_score(va[lab], (p>=t).astype(int), pos_label=1, zero_division=0) for t in THRESHOLDS)
        else:
            sc = 0.0
        if sc > best_score: best_score, best_params, model0 = sc, params, m
        if va is None: break

    # Stage 2: choose threshold + closing on the chosen model's validation probabilities.
    thr, close, val_f1 = 0.5, True, 0.0
    if va is not None and va[lab].nunique() > 1:
        p = model0.predict_proba(va[feats])[:, 1]
        for t in THRESHOLDS:
            base = (p>=t).astype(int)
            for uc in (False, True):
                pred = binary_closing(base, structure=np.ones(5)).astype(int) if uc else base
                f = f1_score(va[lab], pred, pos_label=1, zero_division=0)
                if f > val_f1: val_f1, thr, close = f, t, uc
    return dict(model=model0, feats=feats, thr=thr, close=close, val_f1=val_f1,
                params=best_params, close_size=close_size, min_run=mr,
                prev=float((mdf[lab]==1).mean()))

print('train_motor() ready (grid search over', len(PARAM_GRID), 'configs per motor).')

train_motor() ready (grid search over 24 configs per motor).


### Motor 1

Hardest motor (temperature↔fault inversion).

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| Absolute features (HistGB) | 0.23 |
| + dynamic features | 0.54 |
| + injection peak tuning (8–16°C) | 0.66 |
| + window-shape (skew/kurt/slope/range/EWMA) | 0.72 |

**Best: HistGB with dynamic + window-shape features**, injection peak 8–16°C.

### Motor 2

Frequent fault, easy to separate.

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| Physical + dynamic (HistGB) | 0.90 |
| RandomForest | 0.74 |
| Logistic Reg. | 0.46 |

**Best: HistGB with physical + dynamic features.**

### Motor 3

Extremely rare fault (<0.5%).

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| HistGB + heavy injection (×16) | 0.77 |
| IsolationForest (anomaly) | 0.10 |

**Best: HistGB sustained by abundant subtle injection (peak 3–8°C ×16).**

### Motor 4

Also affected by the inversion.

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| Absolute features | 0.20 |
| + dynamic features | 0.74 |
| + post-processing (close=5, min_run=40) | 0.90 |

**Best: HistGB with dynamic features + aggressive morphological post-processing.**

### Motor 5

Extremely rare fault (<0.5%).

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| HistGB + subtle injection (×16, 1–4°C) | 0.96 |

**Best: HistGB with subtle injection; already excellent.**

### Motor 6

Marked fault, contiguous.

**Summary of the models tried** (F1 of the fault class):

| Model | F1 (fault class) |
|---|---|
| HistGB physical + dynamic | 0.89 |
| + post-processing (min_run=20) | 0.93 |

**Best: HistGB + minimum-run post-processing.**

### Train all six motors and report held-out F1

We fit every motor with its per-motor configuration and report the fault-class F1 on the held-out
validation sequences (an honest, leakage-free estimate on training data).

In [5]:
models = {}
rows = []
for mid in range(1, 7):
    info = train_motor(mid, seed=42)
    models[mid] = info
    rows.append((f'Motor {mid}', info['val_f1'], info['thr'], info['close'],
                 info['close_size'], info['min_run']))
summary = pd.DataFrame(rows, columns=['Motor','Held-out F1 (fault)','Threshold','Closing','close_size','min_run'])
print(summary.to_string(index=False))

  Motor  Held-out F1 (fault)  Threshold  Closing  close_size  min_run
Motor 1             0.473955       0.05    False         NaN        0
Motor 2             0.433514       0.05     True         NaN        0
Motor 3             0.949367       0.70    False         NaN        0
Motor 4             0.720348       0.05     True         5.0       40
Motor 5             0.761538       0.50     True         NaN        0
Motor 6             0.887538       0.30     True         NaN       20


### Prepare final submission

Using exactly the per-motor models selected above, we predict the 8 test sequences, apply the per-motor
decision threshold and morphological post-processing, and write the submission file with 0/1 predictions
for all six motors.

In [6]:
sub = pd.read_csv(os.path.join(ROOT, 'sample_submission.csv'))
final = sub.copy()

def predict_column(mi, thr):
    '''Predict one motor across all test sequences at threshold `thr` (+ post-processing).'''
    col = np.zeros(len(sub), int)
    for tid in sub['test_condition'].unique():
        mask = (sub['test_condition'] == tid).to_numpy()
        td = test_df[test_df['test_condition'] == tid].sort_values('time')
        p = mi['model'].predict_proba(td[mi['feats']])[:, 1]
        y = (p >= thr).astype(int)
        if mi['close_size']: y = binary_closing(y, structure=np.ones(mi['close_size'])).astype(int)
        elif mi['close']:    y = binary_closing(y, structure=np.ones(3)).astype(int)
        if mi['min_run'] > 0: y = remove_short_runs(y, mi['min_run'])
        L = int(mask.sum())
        if len(y) != L: y = (y[:L] if len(y) > L else np.pad(y, (0, L-len(y))))
        col[mask] = y
    return col

# Use exactly the models selected above (trained on the per-sequence training split),
# the same way the winning pipeline produced its predictions.
for mid in range(1, 7):
    mi = models[mid]
    col = predict_column(mi, mi['thr'])
    if col.sum() == 0:                                       # degenerate -> prevalence-floor rescue
        probs = mi['model'].predict_proba(test_df[mi['feats']])[:, 1]
        rank_thr = float(np.quantile(probs, 1.0 - min(max(mi['prev'], 1e-4), 0.5)))
        col = predict_column(mi, rank_thr)
        print(f'  M{mid}: validation threshold gave 0 faults -> prevalence-floor fallback applied')
    final[f'data_motor_{mid}_label'] = col

out = os.path.join(ROOT, 'final_submission_group_X.csv')
final.to_csv(out, index=False)
print('Saved', out, '| shape', final.shape)
print('Predicted faults per motor:',
      {f'M{m}': int((final[f'data_motor_{m}_label']==1).sum()) for m in range(1,7)})
final.head()

Saved C:\Users\oscar\Desktop\TDS INDUSTRY\final_submission_group_X.csv | shape (14157, 8)
Predicted faults per motor: {'M1': 1542, 'M2': 1026, 'M3': 38, 'M4': 354, 'M5': 96, 'M6': 182}


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,0,0,0,0,0,20240527_094865
1,1,1,0,0,0,0,0,20240527_094865
2,2,1,0,0,0,0,0,20240527_094865
3,3,1,0,0,0,0,0,20240527_094865
4,4,1,0,0,0,0,0,20240527_094865


## Discussions and Conclusions

**Scores.** Our best submission reaches a **public macro-F1 of 0.866** on Kaggle, with the per-motor
breakdown M1 0.722 · M2 0.905 · M3 0.769 · M4 0.901 · M5 0.963 · M6 0.934 (obtained from the validator's
per-motor feedback). However, the 0.30437 obtained from the privated leaderboard showed we have clearly overfitted.

**Why the final model overfitted.**
Looking back at our strategy, a few critical factors caused that massive drop on the private leaderboard:
- Over-tuning the Decision Thresholds: We aggressively fine-tuned our decision thresholds for each motor to maximize performance on a very tiny local validation split. When you optimize a threshold on a microscopic sample of rare events, it almost never generalizes to unseen data.
- The Synthetic Data Trap: For the rare-fault motors (M1, M3, M5), our model relied heavily on the synthetic "rise-and-fall" pulses we injected. It's highly likely the model became very good at finding our injected faults, but couldn't recognize the messier, nuanced patterns of the real faults hiding out in the private test set.
- Small Sample Variance: Even though our sequence-by-sequence validation strategy was the correct approach to avoid data leakage, the actual number of sequences we had with real faults was just too small. The validation set simply wasn't diverse enough to represent the private test set.

**Strengths.** 
- Solid, Interpretable Pipeline: We built a fast, tree-based architecture that was easy to explain, quick to train, and allowed for rapid iteration.
- Good Performance on the Hardest Motors: Our feature engineering allowed us to make massive gains on the two most difficult motors.
- Sequence Validation: While limited by sample size, keeping our validation strictly per-sequence successfully protected us from the basic row-wise temporal leakage.

**Limitations.**
- The temperature↔fault **inversion** is a genuine structural ceiling: M1 stays around 0.72 and is the main
  source of variance.
- The synthetic injection is **stochastic**, so the exact macro-F1 depends on the random seed (the
  rare-fault motors M1/M3/M5 can swing noticeably). The 0.866 is our best verified run; the seed-averaged
  expectation is somewhat lower, although the *architecture* (features, model, post-processing) is robust.
- We rely on a single classifier family; temporal models were not fully exploited.

**Potential improvements.**
- **Variance reduction:** seed-ensembling (average several injection seeds) and choosing the decision
  threshold by cross-validation rather than a single held-out split, to remove the occasional collapses.
- **Cross-motor context:** M2 and M4 fail in the same session, so adding M4's dynamics as context for M2
  measurably stabilised M2 — extending this idea is promising.
- **Temporal models** (LSTM/1D-CNN over windows) to capture dependencies the tabular model ignores.
- **Leave-one-fault-sequence-out** validation as a more honest model-selection criterion given only two
  real fault sequences.